# variant monitoring システム（講義用）

このノートブックでは、SARS-CoV-2の変異株ゲノムメタデータを入力として、相対成長率（relative growth rate） と 頻度推移（θ） をベイズ多項ロジスティック回帰（MCMC）で推定します。「どの変異株が集団内で広がりやすいか」を定量化し、各変異株の増加傾向と推定の不確かさを同時に評価することが目的です。結果はPDF・TXTファイルとして出力されます。


**全体の流れ：**  環境構築 → データ読み込み → Stanモデルへのデータ変換 → MCMCサンプリング → 結果の可視化


---



【解説】はじめに、環境構築を行います。

In [ ]:
# =============================================================================
# Cell 1: 環境構築とパッケージのインストール
# =============================================================================

# 不足しているパッケージをインストール
install.packages(c("patchwork", "posterior", "data.table"))

# cmdstanrのインストール（公式リポジトリから）
install.packages("cmdstanr", repos = c("https://mc-stan.org/r-packages/", getOption("repos")))

library(cmdstanr)

# CmdStan自体のインストール（Colab環境にダウンロード・ビルドします。数分かかります）
install_cmdstan(cores = 2)

# CmdStanのパスが通っているか確認
cmdstan_path()

In [ ]:
# =============================================================================
# Cell 2: ライブラリの読み込み
# =============================================================================
library(dplyr)
library(tidyverse)
library(data.table)
library(ggplot2)
library(cmdstanr)
library(patchwork)
library(RColorBrewer)
library(posterior)
library(httr)

In [ ]:
# =============================================================================
# Cell 3: 設定、inputファイル、変数の定義
# =============================================================================
# URLを指定
meta1_file_url <- "https://github.com/virus-informatics/-/raw/fc0f48c0674c7ae6c5bbaa28e66169ed7ae37cdf/input.zip"
meta2_file_url <- "https://github.com/virus-informatics/-/raw/fc0f48c0674c7ae6c5bbaa28e66169ed7ae37cdf/input2.zip"
stan_file_url  <- "https://github.com/virus-informatics/-/raw/fc0f48c0674c7ae6c5bbaa28e66169ed7ae37cdf/multinomial_independent.ver2.stan"

zip1_name <- "input.zip"
zip2_name <- "input2.zip"
stan_f.name <- "multinomial_independent.ver2.stan"

# ファイルのダウンロード
GET(meta1_file_url, write_disk(zip1_name, overwrite = TRUE))
GET(meta2_file_url, write_disk(zip2_name, overwrite = TRUE))
GET(stan_file_url,  write_disk(stan_name, overwrite = TRUE))

# にZIPを解凍
unzip(zip1_name, exdir = "/content")
unzip(zip2_name, exdir = "/content")

**【💡ここで変更が必要】**  解析対象ファイルとパラメータの設定 次のセルでは解析に使うファイル名と分析パラメータを定義します。別の期間やデータで試す場合は、metadata.name.usa の値を変更してください。




In [ ]:
# --- 使用するファイル名に合わせて以下を変更 ---
metadata.name.usa <- "/content/2023-10-01_to_2024-03-31.txt"

# 作業ディレクトリをColabのデフォルトに設定
dir <- "/content/"
setwd(dir)

# --------------------------------------------------------------

# 出力ファイル名の生成
output_date <- gsub(".txt", "", metadata.name.usa)

# 出力用ディレクトリ（無ければ作成）
if(!dir.exists("/content/output")) { dir.create("/content/output") }

pdf.growth.rate.name <- paste0("/content/output/", output_date, ".method1.growth_rate.pdf")
txt.growth.rate.name <- paste0("/content/output/", output_date, ".method1.growth_rate.txt")
pdf.theta.name       <- paste0("/content/output/", output_date, ".method1.theta.pdf")

# 分析パラメータ
region.interest      <- "USA"
core.num             <- 2
limit.count.analyzed <- 30
bin.size             <- 1
generation_time      <- 2.1

【解説】データの前処理

次のセルでは、メタデータをStanモデルに渡せる形に整形します。大きく3つの処理を行います。

① 変異株のフィルタリング
サンプル数が少ない変異株（デフォルト30件未満）は除外します。サンプルが少ないと成長率の推定精度が低く、ノイズになるためです。

② タイムビンへの変換
日付を連続値（1, 2, 3, ...）に変換し、`bin.size` 日ごとに区切ります。これにより「日ごとの変異株カウント行列 Y」が作られます。

③ Stan用データリストの作成
Stanモデルは `X`（時間の計画行列）と `Y`（各日・各変異株のカウント）を受け取ります。`K`（変異株数）、`N`（タイムビン数）なども一緒に渡します。

In [ ]:
# =============================================================================
# Cell 4: データの読み込みとStan用データの作成
# =============================================================================

# Stanモデルのコンパイル
multi_nomial_model <- cmdstan_model(stan_f.name)

# メタデータの読み込み
metadata.filtered.analyzed.region <- fread(
  metadata.name.usa,
  header = TRUE, sep = "\t", quote = "", check.names = TRUE
)

metadata.filtered.analyzed.region <- metadata.filtered.analyzed.region %>%
  rename("Pango.lineage" = "Nextclade_pango")

# データを軽量化
set.seed(42)
metadata.filtered.analyzed.region <- metadata.filtered.analyzed.region %>% sample_n(20000)

# リファレンス変異株の特定
variant.ref <- names(sort(table(metadata.filtered.analyzed.region$Pango.lineage), decreasing = TRUE)[1])
cat("Reference variant:", variant.ref, "\n")

# 十分な配列数のある変異株をフィルタリング
total.analyzed <- nrow(metadata.filtered.analyzed.region)
count.analyzed.region.df <- metadata.filtered.analyzed.region %>%
  group_by(pango_lineage) %>%
  summarize(count.analyzed = n(), prop.analyzed = count.analyzed / total.analyzed)

count.analyzed.region.df.filtered <- count.analyzed.region.df %>%
  filter(count.analyzed > limit.count.analyzed) %>%
  arrange(desc(count.analyzed))

variant.interest.v <- as.character(count.analyzed.region.df.filtered$pango_lineage)

metadata.filtered.analyzed.region.selected <- metadata.filtered.analyzed.region %>%
  filter(pango_lineage %in% variant.interest.v)

# タイムビンの作成
metadata.filtered.interest <- metadata.filtered.analyzed.region.selected %>%
  mutate(
    date.num     = as.numeric(date) - min(as.numeric(date)) + 1,
    date.bin     = cut(date.num, seq(0, max(date.num), bin.size)),
    date.bin.num = as.numeric(date.bin)
  ) %>%
  filter(!is.na(date.bin))

# Stan用マトリクスの構築
metadata.filtered.interest.bin <- metadata.filtered.interest %>%
  group_by(date.bin.num, pango_lineage) %>%
  summarize(count = n(), .groups = "drop")

metadata.filtered.interest.bin.spread <- metadata.filtered.interest.bin %>%
  spread(key = pango_lineage, value = count)

metadata.filtered.interest.bin.spread[is.na(metadata.filtered.interest.bin.spread)] <- 0

X <- as.matrix(data.frame(
  X0 = 1,
  X1 = metadata.filtered.interest.bin.spread$date.bin.num
))

Y <- metadata.filtered.interest.bin.spread %>%
  select(-date.bin.num) %>%
  select(all_of(variant.interest.v)) %>%
  as.matrix()

group.df <- data.frame(group_Id = 1:ncol(Y), group = colnames(Y))
Y_sum.v <- apply(Y, 1, sum)

# Stanに渡すデータリスト
data.stan <- list(
  K = ncol(Y), N = nrow(Y), D = 2,
  X = X, Y = Y,
  generation_time = generation_time,
  bin_size = bin.size,
  Y_sum = Y_sum.v
)

【解説】ベイズ推定とMCMCとは？

このモデルでは「変異株 *k* の成長率 *rk* ​」をパラメータとして、観測データ（カウント行列 *Y*）から事後分布を推定します。

MCMCサンプリング（マルコフ連鎖モンテカルロ法） とは、複雑な事後分布を直接解析的に解く代わりに、確率的なサンプリングによって近似する手法です。

* `iter_warmup`：サンプラーがパラメータ空間を探索する「ウォームアップ」期間。この分は捨てられます
* `iter_sampling`：実際に使うサンプル数。多いほど推定が安定しますが時間がかかります
* `chains`：独立した連鎖数。複数で同じ結果が得られれば収束の証拠になります

⏱ このセルは数分〜十数分かかります。 実行中は進捗ログが流れます。`Rhat` が 1.01 以下であれば収束していると判断できます。

In [ ]:
# =============================================================================
# Cell 5: Bayesianモデルの実行 (MCMC Sampling)
# =============================================================================
fit.stan <- multi_nomial_model$sample(
  data           = data.stan,
  iter_sampling  = 1000,
  iter_warmup    = 500,
  seed           = 1234,
  parallel_chains = 2, # Colab無料枠のCPUコア数に合わせて2に設定
  max_treedepth  = 20,
  chains         = core.num
)

【解説】結果の読み方

MCMCが完了すると、各パラメータの事後分布のサンプルが得られます。次のセルでは主に2つのパラメータを可視化します。

① `growth_rate`（相対成長率）
最も多い変異株（リファレンス）を基準（= 1）としたときの各変異株の相対的な増加速度です。1より大きいほど「リファレンスより速く広がっている」ことを意味します。エラーバーは95%信頼区間です。

② `theta`（頻度推移）
各タイムビンにおける各変異株の頻度（割合）の事後分布です。実測頻度（点）とモデル推定値（リボン）を重ねて表示することで、モデルの当てはまりを視覚的に確認できます。

In [ ]:
# =============================================================================
# Cell 6: 成長率の抽出とプロット・保存
# =============================================================================

# --- 成長率のサマリー ---
stat.info <- fit.stan$summary("growth_rate") %>% as.data.frame()
stat.info$Pango.lineage <- colnames(Y)[2:ncol(Y)]

stat.info.q <- fit.stan$summary("growth_rate", ~quantile(.x, probs = c(0.005, 0.995))) %>%
  as.data.frame() %>% rename(q0.5 = `0.5%`, q99.5 = `99.5%`)

stat.info <- stat.info %>%
  inner_join(stat.info.q, by = "variable") %>%
  mutate(signif = ifelse(q0.5 > 1, 'higher', 'not higher')) %>%
  arrange(mean) %>%
  mutate(Pango.lineage = factor(Pango.lineage, levels = Pango.lineage))

stat.info_10 <- stat.info %>% top_n(10, mean) %>% arrange(desc(mean))

# --- プロット1 (Bar Chart) ---
ylabel <- paste0('relative growth rate per generation (per ', variant.ref, ')')
g <- ggplot(stat.info_10, aes(x = Pango.lineage, y = mean, fill = signif)) +
  geom_bar(stat = "identity") +
  geom_errorbar(aes(ymin = q5, ymax = q95), width = 0.2) +
  geom_hline(yintercept = 1, linetype = "dashed", alpha = 0.5) +
  coord_flip() +
  scale_fill_manual(breaks = c("higher", "not higher"), values = c("lightsalmon", "gray70")) +
  xlab('') + ylab(ylabel) + ggtitle(region.interest) +
  theme_classic(base_size = 12, base_family = "Helvetica") +
  theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
        panel.background = element_blank(), strip.text = element_text(size = 8))

# Colab上で画像を表示
print(g)

# --- 推移プロットの準備 ---
date.start <- min(metadata.filtered.interest$date)
data_Id.df <- data.frame(
  date_Id = X[, 2],
  Y_sum   = apply(as.data.frame(Y), 1, sum),
  date    = as.Date(X[, 2], origin = date.start)
)

data.freq <- metadata.filtered.interest.bin %>%
  rename(group = pango_lineage) %>% filter(group != "others") %>%
  group_by(date.bin.num) %>% mutate(freq = count / sum(count)) %>%
  merge(data_Id.df, by.x = "date.bin.num", by.y = "date_Id")

draw.df.theta <- fit.stan$draws("theta", format = "df") %>% as.data.frame() %>%
  select(!contains('.')) %>% sample_frac(size = 0.5)

draw.df.theta.long <- draw.df.theta %>% gather(key = class, value = value) %>%
  mutate(
    date_Id  = str_match(class, 'theta\\[([0-9]+),[0-9]+\\]')[, 2] %>% as.numeric(),
    group_Id = str_match(class, 'theta\\[[0-9]+,([0-9]+)\\]')[, 2] %>% as.numeric()
  ) %>% inner_join(data_Id.df, by = "date_Id")

draw.df.theta.long.sum <- draw.df.theta.long %>%
  group_by(group_Id, date) %>%
  summarize(mean = mean(value), ymin = quantile(value, 0.025), ymax = quantile(value, 0.975), .groups = "drop") %>%
  inner_join(group.df, by = "group_Id") %>%
  inner_join(data.freq %>% select(group, count, freq, date), by = c("date", "group"))

variant.top.v.wo_ref <- stat.info %>% top_n(5, mean) %>% arrange(desc(mean)) %>%
  pull(Pango.lineage) %>% as.character()

draw.df.theta.long.sum.filtered <- draw.df.theta.long.sum %>%
  filter(group %in% variant.top.v.wo_ref) %>% unique() %>%
  mutate(group = factor(group, levels = variant.top.v.wo_ref))

# --- プロット2 (Trajectory Plot) ---
g2 <- ggplot(draw.df.theta.long.sum.filtered, aes(x = date, y = mean, fill = group, color = group)) +
  geom_point(aes(y = freq, size = count), alpha = 0.4) +
  geom_ribbon(aes(ymin = ymin, ymax = ymax), color = NA, alpha = 0.4) +
  geom_line(linewidth = 0.3) +
  scale_x_date(date_labels = "%m-%d", date_breaks = "2 weeks", date_minor_breaks = "1 week") +
  scale_color_brewer(palette = "Dark2") + scale_fill_brewer(palette = "Dark2") +
  scale_size_continuous(range = c(0.5, 5)) + ggtitle(region.interest) +
  theme_classic(base_size = 12, base_family = "Helvetica") +
  theme(panel.grid.major = element_blank(), panel.grid.minor = element_blank(),
        panel.background = element_blank(), strip.text = element_text(size = 8))

# Colab上で画像を表示
print(g2)

# --- 出力ファイルの保存 ---
write.table(stat.info, txt.growth.rate.name, col.names = TRUE, row.names = FALSE, sep = "\t", quote = FALSE)

pdf(pdf.growth.rate.name, width = 5, height = 6)
print(g)
dev.off()

pdf(pdf.theta.name, width = 6, height = 6.5)
print(g2)
dev.off()

cat("Analysis complete. Outputs saved to:\n")
cat(" ", pdf.growth.rate.name, "\n")
cat(" ", txt.growth.rate.name, "\n")
cat(" ", pdf.theta.name, "\n")